# ODE Driver

This notebook loads `config.yaml`, defines a system, solves it using the helper library in `Helpers/`, and creates phase-space and Poincaré plots.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Helpers.config_loader import load_config
from Helpers.ic_generator import generate_ic_grid, get_ic_grid_info
from Helpers.ode_viewer_nD import ODESystemND, ODEViewerND

config = load_config('config.yaml')
system_config = config.get('system', {})
integration_config = config.get('integration', {})
plot_3d_config = config.get('plot_3d', {})
poincare_config = config.get('poincare', {})
animation_config = config.get('animation', {})
ic_config = config.get('ic_grid', {})

N_VARS = int(system_config.get('n_vars', 3))
FIRST_ORDER_SYSTEM = bool(system_config.get('first_order', True))
VAR_NAMES = system_config.get('var_names', [f'x_{i}' for i in range(1, N_VARS + 1)])

t_start = float(integration_config.get('t_start', 0.0))
t_end = float(integration_config.get('t_end', 300.0))
num_points = int(integration_config.get('num_points', 3000))
step_size = float(integration_config.get('step_size', 1e-3))
compute_boundary = integration_config.get('compute_boundary', None)
if compute_boundary is not None:
    compute_boundary = float(compute_boundary)

IC_CENTER = ic_config.get('center', [0.0] * N_VARS)
IC_SPREAD = ic_config.get('spread', 1.0)
ICS_PER_VAR = ic_config.get('ics_per_var', [8] * N_VARS)

projection_axes = plot_3d_config.get('projection_axes', [0, 1, 3])
color_by = plot_3d_config.get('color_by', 2)
figsize = tuple(plot_3d_config.get('figsize', [6, 6]))
title = plot_3d_config.get('title', 'Phase Space')
save_path = plot_3d_config.get('save_path', './Plots/3D/3D_NSYS_ODE.png')
colormap = plot_3d_config.get('colormap', 'viridis')

poincare_enabled = bool(poincare_config.get('enabled', False))
poincare_plane = tuple(poincare_config.get('plane', [1, 0, 0, -3]))
poincare_point_size = float(poincare_config.get('point_size', 0.6))
poincare_color_by = color_by if poincare_config.get('flip_x', False) is False else None

animation_enabled = bool(animation_config.get('enabled', False))
animation_start_plane = tuple(animation_config.get('start_plane', [1, 0, 0, -0.01]))
animation_end_plane = tuple(animation_config.get('end_plane', [1, 0, 0, -299.99]))
animation_save = animation_config.get('poincare_animation_save', './Plots/2D/poincare_animation.html')
animation_duration = float(animation_config.get('poincare_animation_duration', 15))
animation_fps = int(animation_config.get('poincare_animation_fps', 24))
animation_trail = int(animation_config.get('poincare_animation_trail', 3))

print('Config loaded successfully')
print('N_VARS:', N_VARS, 'VAR_NAMES:', VAR_NAMES)
print('Projection axes:', projection_axes, 'color_by:', color_by)

In [ ]:
# Define your system here
# For a first-order system, return a list of N_VARS derivatives.

def system_func(t, state):
    x, y, z = state
    dx = 0.3 * x - 0.4 * y * z - 0.15 * x
    dy = 0.3 * y + 0.2 * z - 0.3 * y
    dz = 1.0 * x - 0.5 * y - 0.2 * z
    return [dx, dy, dz]

# If using a higher-order system, uncomment and define highest_derivative instead.
# def highest_derivative(t, state):
#     return ...

In [ ]:
# Build the ODE system and solve the initial-condition grid

ic_grid = generate_ic_grid(N_VARS, IC_CENTER, IC_SPREAD, ICS_PER_VAR)
total_ics = get_ic_grid_info(N_VARS, ICS_PER_VAR)

print(f'Using {len(ic_grid)} initial conditions')

if FIRST_ORDER_SYSTEM:
    system = ODESystemND(system_func, N_VARS, VAR_NAMES)
else:
    raise NotImplementedError('Higher-order systems require `highest_derivative` and custom conversion.')

t_eval = np.linspace(t_start, t_end, num_points)
viewer = ODEViewerND()

viewer.solve_ics_grid(
    system,
    (t_start, t_end),
    ic_grid,
    total_ics,
    t_eval=t_eval,
    max_step=step_size,
)

print(f'Solved {len(viewer.solutions)} trajectories')

In [ ]:
# Plot the 3D phase-space trajectories
viewer.plot_all_3d(
    projection_axes=projection_axes,
    color_variable=color_by,
    figsize=figsize,
    title=title,
    save_path=save_path,
    compute_boundary=compute_boundary,
)

In [ ]:
# Compute and display the Poincaré section
if poincare_enabled:
    intersections = viewer.compute_poincare_intersections(
        projection_axes=projection_axes,
        plane=poincare_plane,
        compute_boundary=compute_boundary,
        color_variable=color_by,
    )
    print(f'Found {len(intersections)} Poincaré intersections')
    viewer.plot_poincare_section(
        projection_axes=projection_axes,
        plane=poincare_plane,
        compute_boundary=compute_boundary,
        color_variable=color_by,
        figsize=figsize,
        title='Poincaré Section',
        save_path='./Plots/2D/poincare_section.png',
    )

if animation_enabled:
    from Helpers.poincare_animation import save_poincare_animation

    save_poincare_animation(
        viewer,
        start_plane=animation_start_plane,
        end_plane=animation_end_plane,
        projection_axes=projection_axes,
        duration=animation_duration,
        fps=animation_fps,
        trail_length=animation_trail,
        compute_boundary=compute_boundary,
        color_variable=color_by,
        figsize=figsize,
        save_path=animation_save,
    )